In [ ]:
import sys
sys.path.append('../')

import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb
import numpy as np
from utils import * #only needed for xgboost
import matplotlib.pyplot as plt
from baselines import *

import forest_config as c

In [ ]:

for d in c.d_list:
    ablation_transfer_real = pd.DataFrame(columns = ['seed', 'target_column', 'target_instances', 'method',
                                   'v', 'target_tree_size', 'val_rmse', 'val_mae', 'rmse', 'mae'])

    for seed in c.seed_list:
        for train_size in c.train_size_list:
            for target_column in c.target_columns:
            

                #data from Svedala
                data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])
                data_sweden = data_sweden[data_sweden['area_code'] == d]
                data_sweden = data_sweden.sample(1000, random_state=seed) #random sample of 1000 source instances


                #evaluate and rain on latvia instead (keep naming for simplicity)
                #data from latvia target
                data_latvia = pd.read_csv(r'../datasets/rs_lettland.csv', index_col=[0])
                data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
                data_temp, data_test = train_test_split(data_latvia, test_size=0.25, random_state=seed)
                data_train, data_val = train_test_split(data_temp, test_size=0.333, random_state=seed)
                train_size_ = int(len(data_train)*train_size)
                data_train = data_train[0:train_size_]

                #"General" base dataset (to use for transfer)
                X_source_train = np.array(data_sweden[c.predictor_columns])
                y_source_train = np.array(data_sweden[target_column]) #change this to "Dgv" to use diameter as source label!

                #Specific train and test set
                X_target_train = np.array(data_train[c.predictor_columns])
                y_target_train = np.array(data_train[target_column])

                X_target_val = np.array(data_val[c.predictor_columns])
                y_target_val = np.array(data_val[target_column])

                X_target_test = np.array(data_test[c.predictor_columns])
                y_target_test = np.array(data_test[target_column])

                #additional source/target dummy sets for naive xgboost
                ones = np.ones((len(X_target_train), 1))
                X_target_train_with_dummy = np.hstack((X_target_train, ones))
                ones = np.ones((len(X_target_val), 1))
                X_target_val_with_dummy = np.hstack((X_target_val, ones))
                ones = np.ones((len(X_target_test), 1))
                X_target_test_with_dummy= np.hstack((X_target_test, ones))

                zeros = np.zeros((len(X_source_train), 1))
                X_source_train_with_dummy = np.hstack((X_source_train.copy(), zeros))

                print(len(X_target_train), len(X_target_val), len(X_target_test))
                for config in c.param_grid_XGBoost:
                    v, target_tree_size = config



                    #We also append baseline results!!
                    method = 'xgboost'
                    params = {
                        'objective': 'reg:squarederror',  # Regression with squared error
                        'max_depth': target_tree_size,                   # Maximum depth of a tree
                        'eta': v,                       # Learning rate
                        'eval_metric': 'rmse',           # RMSE as evaluation metric
                        }
                            
                    bst = train_xgboost(X_target_train, y_target_train, X_target_val, y_target_val, boosting_rounds=1000, params=params)
                    preds = test_xgboost(X_target_test, bst)
                    val_preds = test_xgboost(X_target_val, bst)
                    val_rmse = compute_rmse(val_preds, y_target_val)
                    val_mae = compute_mae(val_preds, y_target_val)
                    rmse = compute_rmse(preds, y_target_test)
                    mae = compute_mae(preds, y_target_test)
                    ablation_transfer_real.loc[len(ablation_transfer_real)] = [seed, target_column, train_size_, method, v, target_tree_size, 
                                                                            val_rmse, val_mae, rmse, mae]
                    
                    ablation_transfer_real.to_csv(f'results/xgboost_ablation_HGV_rs_{d}.csv') #if using Dgv as source label change HGV to DGV!


                    method = 'xgboost_naive_transfer_with_dummy'
                    params = {
                        'objective': 'reg:squarederror',  # Regression with squared error
                        'max_depth': target_tree_size,                   # Maximum depth of a tree
                        'eta': v,                       # Learning rate
                        'eval_metric': 'rmse',           # RMSE as evaluation metric
                        }
                    X_comb = np.concatenate((X_target_train_with_dummy, X_source_train_with_dummy)) 
                    y_comb = np.concatenate((y_target_train, y_source_train))       
                    bst = train_xgboost(X_comb, y_comb, X_target_val_with_dummy, y_target_val, boosting_rounds=1000, params=params)
                    preds = test_xgboost(X_target_test_with_dummy, bst)
                    val_preds = test_xgboost(X_target_val_with_dummy, bst)
                    val_rmse = compute_rmse(val_preds, y_target_val)
                    val_mae = compute_mae(val_preds, y_target_val)
                    rmse = compute_rmse(preds, y_target_test)
                    mae = compute_mae(preds, y_target_test)
                    ablation_transfer_real.loc[len(ablation_transfer_real)] = [seed, target_column, train_size_, method, v, target_tree_size, 
                                                                            val_rmse, val_mae, rmse, mae]
                    
                    ablation_transfer_real.to_csv(f'results/xgboost_ablation_HGV_rs_{d}.csv') #if using Dgv as source label change HGV to DGV!